
# Imports and Loading Dataset - PHASE 1

In [24]:
import numpy as np
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def load_data():
    """ load prices.csv into dataframe
        path is located as per the user's system
    """
    data_path = Path.cwd().parent / "data" / "prices.csv"
    df = pd.read_csv(data_path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = pd.to_datetime(df["date"])
    return df

df = load_data()
print("prices.csv loaded successfully.")

prices.csv loaded successfully.


## LOG 1   :

    - The parse_dates=["date"] fails , while  \usepackage{ulem}  is attached or prepended to the the latex header — the real column is named '\\usepackage{ulem}date', not 'date'.
    - genuinely odd data corruption , My guess: somewhere in this file's history it got pasted through a LaTeX-aware editor/tool that injected formatting markup into the first cell of it .
    - Fix — rename the first column explicitly rather than relying on parse_dates to find "date" by name:

# EDA - Exploring Data Analysis


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26100 entries, 0 to 26099
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    26100 non-null  datetime64[ns]
 1   isin    26100 non-null  object        
 2   ticker  26100 non-null  object        
 3   name    26100 non-null  object        
 4   open    26077 non-null  float64       
 5   high    26077 non-null  float64       
 6   low     26077 non-null  float64       
 7   close   26077 non-null  float64       
 8   volume  26077 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 1.8+ MB


## LOG 2   :

    - Missing values in OHLCV columns — open/high/low/close/volume show 26,077 non-null out of 26,100 rows, so their are 23 missing values scattered somewhere.ut thet other like date, isin, ticker, name are all fully populated.

In [26]:
df.head()

,date,isin,ticker,name,open,high,low,close,volume
0,2021-01-01,XX0000000515,3M,3M,193.8501,197.5727,191.8432,195.0012,6074404.0
1,2021-01-04,XX0000000515,3M,3M,194.9480,196.4742,192.0129,194.1845,3485457.0
2,2021-01-05,XX0000000515,3M,3M,193.5205,194.7041,191.9887,193.6376,2940900.0
3,2021-01-06,XX0000000515,3M,3M,193.7703,195.3852,193.1063,195.1351,2643272.0
4,2021-01-07,XX0000000515,3M,3M,193.8540,194.4080,191.6080,192.6929,1804643.0


In [27]:
df.describe()

,date,open,high,low,close,volume
count,26100,26077.000000,26077.000000,26077.000000,26077.000000,2.607700e+04
mean,2021-07-02 19:07:35.172413696,206.740397,211.091789,203.821193,208.139680,2.020188e+07
min,2021-01-01 00:00:00,0.544200,0.551800,0.536900,0.550000,2.333000e+03
25%,2021-04-02 00:00:00,86.764200,88.032100,85.446600,86.780200,9.281119e+06
50%,2021-07-02 00:00:00,160.010000,162.577300,157.516600,160.234800,1.792378e+07
75%,2021-10-01 00:00:00,275.319400,280.072600,271.251900,276.187700,2.842712e+07
max,2021-12-31 00:00:00,1797.064700,6632.380800,1778.630800,6502.334100,1.352467e+08
std,NaN,168.078413,178.760994,165.929170,176.423116,1.397297e+07


## LOG 3 -
    - 1 . likely only likely  badtick outier in Column "High" high and close maxes are roughly 3.5x higher than open/low maxes , suspicious spike relative to the 75th percentile (~276-280) , should check that specific row and check if it's a genuine data error or not.
    - 2 . extreme low "minimums" - mins are all around .54 to .55.
        thats a huge range for one dataset . maybe accordingly also it could be one stock that have a cheaper price , or could be a data glitch. worth checking

# Checking outliers

In [28]:
# row driving the extreme high value
print(df.loc[df["high"].idxmax()])

date      2021-11-15 00:00:00
isin             XX0000000077
ticker                   MICR
name                Microsoft
open                 593.9942
high                6632.3808
low                  586.1877
close               6502.3341
volume             29145217.0
Name: 15886, dtype: object


Microsoft, 2021-11-15 — this is a data error . open and low sit in a normal, tight range of 586–594. But high and close are roughly 11x larger. which hints like a fat finger or formatting error specifically in high and close, not open/low

the math - dividing 6632.38 by 10 , 663 something and also close value 650 something which makes sense

In [29]:
print(df.loc[df["close"].idxmin()])

date      2021-01-01 00:00:00
isin             XX0000000044
ticker                   NOKI
name                    Nokia
open                   0.5442
high                   0.5518
low                    0.5369
close                    0.55
volume                 4775.0
Name: 17487, dtype: object


 Nokia, 2021-01-01 — likely fine, but check volume , first day of dataset , so can be possible and also the volume for stock is very high but we can confirm by


In [30]:
# Check Nokia's price/volume trend over its first ~10 trading days
df[df["ticker"] == "NOKI"].sort_values("date").head(10)

,date,isin,ticker,name,open,high,low,close,volume
17487,2021-01-01,XX0000000044,NOKI,Nokia,0.5442,0.5518,0.5369,0.5500,4775.0
17488,2021-01-04,XX0000000044,NOKI,Nokia,0.5539,0.5639,0.5493,0.5571,4920.0
17489,2021-01-05,XX0000000044,NOKI,Nokia,0.5542,0.5784,0.5521,0.5715,6120.0
17490,2021-01-06,XX0000000044,NOKI,Nokia,0.5760,0.5821,0.5720,0.5764,3297.0
17491,2021-01-07,XX0000000044,NOKI,Nokia,0.5784,0.6199,0.5742,0.6173,4071.0
17492,2021-01-08,XX0000000044,NOKI,Nokia,0.6143,0.6480,0.6116,0.6410,3258.0
17493,2021-01-11,XX0000000044,NOKI,Nokia,0.6477,0.6564,0.6442,0.6562,2916.0
17494,2021-01-12,XX0000000044,NOKI,Nokia,0.6572,0.6790,0.6499,0.6746,5332.0
17495,2021-01-13,XX0000000044,NOKI,Nokia,0.6726,0.6948,0.6714,0.6915,4989.0
17496,2021-01-14,XX0000000044,NOKI,Nokia,0.6950,0.7132,0.6919,0.7066,3551.0


hence , Nokia is not an outlier coz of steady climb of price


In [31]:
df[df["ticker"] == "MICR"].sort_values("date").query("date >= '2021-11-10' and date <= '2021-11-20'")

,date,isin,ticker,name,open,high,low,close,volume
15883,2021-11-10,XX0000000077,MICR,Microsoft,588.7482,597.8918,584.6258,594.5153,27931612.0
15884,2021-11-11,XX0000000077,MICR,Microsoft,598.1131,601.3202,582.2125,586.4677,25083606.0
15885,2021-11-12,XX0000000077,MICR,Microsoft,585.0383,601.7332,578.6374,591.6741,29660758.0
15886,2021-11-15,XX0000000077,MICR,Microsoft,593.9942,6632.3808,586.1877,6502.3341,29145217.0
15887,2021-11-16,XX0000000077,MICR,Microsoft,593.3910,613.1028,592.7211,606.1085,17847117.0
15888,2021-11-17,XX0000000077,MICR,Microsoft,609.6720,614.8977,592.2430,604.4334,33820251.0
15889,2021-11-18,XX0000000077,MICR,Microsoft,603.3254,614.2762,592.3617,613.1015,31688246.0
15890,2021-11-19,XX0000000077,MICR,Microsoft,614.4742,615.0865,603.1237,609.8704,20612490.0


 so for microsoft we can confirm that the row is an outlier If you divide the bad values by 10: 6632.38 / 10 = 663.24 and 6502.33 / 10 = 650.23,both fit right into this range, consistent with the days on either side. open (593.99) and low (586.19) on the 15th were already correct and untouched, which lines up with the dataset.

## LOG 3 Fix
       divided the two affected columns by 10, restoring consistency with the neighboring data.
        this date isn't inside  June 1 – Oct 13 analysis window, so it wouldn't have affected final top-5 answer directly — but it's still worth documenting this kind of check that could matter if the anomaly had landed


In [32]:
mask = (df["ticker"] == "MICR") & (df["date"] == "2021-11-15")
df.loc[mask, ["high", "close"]] = df.loc[mask, ["high", "close"]] / 10

In [33]:
df.describe()

,date,open,high,low,close,volume
count,26100,26077.000000,26077.000000,26077.000000,26077.000000,2.607700e+04
mean,2021-07-02 19:07:35.172413696,206.740397,210.862885,203.821193,207.915264,2.020188e+07
min,2021-01-01 00:00:00,0.544200,0.551800,0.536900,0.550000,2.333000e+03
25%,2021-04-02 00:00:00,86.764200,88.032100,85.446600,86.780200,9.281119e+06
50%,2021-07-02 00:00:00,160.010000,162.577300,157.516600,160.234800,1.792378e+07
75%,2021-10-01 00:00:00,275.319400,280.072600,271.251900,276.187700,2.842712e+07
max,2021-12-31 00:00:00,1797.064700,2995.010300,1778.630800,2936.284600,1.352467e+08
std,NaN,168.078413,174.304389,165.929170,172.085082,1.397297e+07


again we can see that the fix worked but the outlier still exist in high and close ,high should be close to open/low/close on the same row, not 1.6-1.7x off. This could be  - either , a second small magnitude error  , or high/low spread of volatile stock

In [34]:
print(df.loc[df["high"].idxmax()])

date      2021-06-07 00:00:00
isin             XX0000000119
ticker                   INTE
name                    Intel
open                 268.9697
high                2995.0103
low                  266.2128
close               2936.2846
volume              6461099.0
Name: 12117, dtype: object


In [35]:
print(df.loc[df["close"].idxmax()])

date      2021-06-07 00:00:00
isin             XX0000000119
ticker                   INTE
name                    Intel
open                 268.9697
high                2995.0103
low                  266.2128
close               2936.2846
volume              6461099.0
Name: 12117, dtype: object


again the same pattern , 1.6-1.7x off  , we can divide by 10 again to restore consistency.

In [36]:
df[df["ticker"] == "INTE"].sort_values("date").query("date >= '2021-06-02' and date <= '2021-06-12'")

,date,isin,ticker,name,open,high,low,close,volume
12114,2021-06-02,XX0000000119,INTE,Intel,258.1525,264.6409,257.6831,261.7480,7893154.0
12115,2021-06-03,XX0000000119,INTE,Intel,261.3315,271.4969,260.8427,270.3821,8443666.0
12116,2021-06-04,XX0000000119,INTE,Intel,271.2272,273.8963,266.7508,269.5230,6033727.0
12117,2021-06-07,XX0000000119,INTE,Intel,268.9697,2995.0103,266.2128,2936.2846,6461099.0
12118,2021-06-08,XX0000000119,INTE,Intel,264.9352,268.4654,261.0920,265.2824,8446573.0
12119,2021-06-09,XX0000000119,INTE,Intel,265.4529,265.7326,264.3066,265.4850,4691799.0
12120,2021-06-10,XX0000000119,INTE,Intel,262.6665,269.7965,257.9872,265.6919,6120564.0
12121,2021-06-11,XX0000000119,INTE,Intel,263.9691,269.8152,262.2902,265.1298,6190836.0


In [37]:
mask = (df["ticker"] == "INTE") & (df["date"] == "2021-06-07")
df.loc[mask, ["high", "close"]] = df.loc[mask, ["high", "close"]] / 10

In [38]:
df.describe()

,date,open,high,low,close,volume
count,26100,26077.000000,26077.000000,26077.000000,26077.000000,2.607700e+04
mean,2021-07-02 19:07:35.172413696,206.740397,210.759517,203.821193,207.813923,2.020188e+07
min,2021-01-01 00:00:00,0.544200,0.551800,0.536900,0.550000,2.333000e+03
25%,2021-04-02 00:00:00,86.764200,88.032100,85.446600,86.780200,9.281119e+06
50%,2021-07-02 00:00:00,160.010000,162.577300,157.516600,160.234800,1.792378e+07
75%,2021-10-01 00:00:00,275.319400,280.072600,271.251900,276.187700,2.842712e+07
max,2021-12-31 00:00:00,1797.064700,2543.636200,1778.630800,2493.761000,1.352467e+08
std,NaN,168.078413,173.450413,165.929170,171.254412,1.397297e+07


In [39]:
print(df.loc[df["high"].idxmax()])


date      2021-02-19 00:00:00
isin             XX0000000283
ticker                   COCA
name                Coca-Cola
open                 230.7177
high                2543.6362
low                   226.527
close                2493.761
volume             10868832.0
Name: 6560, dtype: object


In [40]:
print(df.loc[df["close"].idxmax()])

date      2021-02-19 00:00:00
isin             XX0000000283
ticker                   COCA
name                Coca-Cola
open                 230.7177
high                2543.6362
low                   226.527
close                2493.761
volume             10868832.0
Name: 6560, dtype: object


another instance of 10x stock price , coca-cola

In [41]:
df[df["ticker"] == "COCA"].sort_values("date").query("date >= '2021-02-15' and date <= '2021-02-24'")

,date,isin,ticker,name,open,high,low,close,volume
6556,2021-02-15,XX0000000283,COCA,Coca-Cola,233.1586,234.7236,227.6292,228.4705,11682958.0
6557,2021-02-16,XX0000000283,COCA,Coca-Cola,227.6012,228.1958,226.4192,226.7513,18340069.0
6558,2021-02-17,XX0000000283,COCA,Coca-Cola,226.2805,230.6630,219.9673,225.2713,17575691.0
6559,2021-02-18,XX0000000283,COCA,Coca-Cola,225.9324,232.5868,225.1198,229.6048,10761427.0
6560,2021-02-19,XX0000000283,COCA,Coca-Cola,230.7177,2543.6362,226.5270,2493.7610,10868832.0
6561,2021-02-22,XX0000000283,COCA,Coca-Cola,225.7888,226.9093,218.2447,220.6755,6936500.0
6562,2021-02-23,XX0000000283,COCA,Coca-Cola,220.4151,224.2063,218.2419,223.4159,15614127.0
6563,2021-02-24,XX0000000283,COCA,Coca-Cola,223.1536,228.4350,221.6259,227.2994,15233763.0


In [42]:
mask = (df["ticker"] == "COCA") & (df["date"] == "2021-02-19")
df.loc[mask, ["high", "close"]] = df.loc[mask, ["high", "close"]] / 10

In [43]:
df.describe()

,date,open,high,low,close,volume
count,26100,26077.000000,26077.000000,26077.000000,26077.000000,2.607700e+04
mean,2021-07-02 19:07:35.172413696,206.740397,210.671728,203.821193,207.727855,2.020188e+07
min,2021-01-01 00:00:00,0.544200,0.551800,0.536900,0.550000,2.333000e+03
25%,2021-04-02 00:00:00,86.764200,88.032100,85.446600,86.780200,9.281119e+06
50%,2021-07-02 00:00:00,160.010000,162.577300,157.516600,160.234800,1.792378e+07
75%,2021-10-01 00:00:00,275.319400,280.055700,271.251900,276.187400,2.842712e+07
max,2021-12-31 00:00:00,1797.064700,2253.991500,1778.630800,2209.795600,1.352467e+08
std,NaN,168.078413,172.847914,165.929170,170.668496,1.397297e+07


 instead of going single rows , i will make scanner for large disprop... errors

In [44]:
# Flag any row where high or close looks disproportionately large vs open/low
suspect = df[(df["high"] > df[["open", "low"]].max(axis=1) * 3) |
             (df["close"] > df[["open", "low"]].max(axis=1) * 3)]
print(suspect)
print(f"\nTotal suspect rows: {len(suspect)}")

            date          isin ticker        name      open       high  \
1012  2021-11-18  XX0000000903    ATT        AT&T   32.7097   366.1413   
2511  2021-08-17  XX0000000093   AMAZ      Amazon  124.5626  1384.4851   
3142  2021-01-15  XX0000000069   APPL       Apple  205.1180  2253.9915   
6167  2021-08-19  XX0000000135   CISC       Cisco   62.9867   720.7191   
9292  2021-08-10  XX0000000853   EXXO  ExxonMobil  198.2216  2204.4001   
9778  2021-06-21  XX0000000473   FORD        Ford  203.9702  2229.6949   
18973 2021-09-13  XX0000000739   PFIZ      Pfizer   90.6789  1005.1814   

            low      close      volume  
1012    32.2685   358.9620  24209977.0  
2511   121.8982  1357.3384  44120069.0  
3142   198.5644  2209.7956  40282460.0  
6167    62.4909   706.5874  50781448.0  
9292   195.6429  2161.1766  32430933.0  
9778   194.9864  2185.9754   2193271.0  
18973   88.7489   985.4719  22751729.0  

Total suspect rows: 7


so we have 7 more outlier rows , we can see , with the past total 10
we can fix by diving by 10

In [45]:
mask = (df["high"] > df[["open", "low"]].max(axis=1) * 3) | \
       (df["close"] > df[["open", "low"]].max(axis=1) * 3)

df.loc[mask, ["high", "close"]] = df.loc[mask, ["high", "close"]] / 10

print(f"Rows corrected: {mask.sum()}")

Rows corrected: 7


In [46]:
df.describe()

,date,open,high,low,close,volume
count,26100,26077.000000,26077.000000,26077.000000,26077.000000,2.607700e+04
mean,2021-07-02 19:07:35.172413696,206.740397,210.320915,203.821193,207.383921,2.020188e+07
min,2021-01-01 00:00:00,0.544200,0.551800,0.536900,0.550000,2.333000e+03
25%,2021-04-02 00:00:00,86.764200,88.026400,85.446600,86.765300,9.281119e+06
50%,2021-07-02 00:00:00,160.010000,162.542500,157.516600,160.188100,1.792378e+07
75%,2021-10-01 00:00:00,275.319400,279.882700,271.251900,275.933900,2.842712e+07
max,2021-12-31 00:00:00,1797.064700,1809.869300,1778.630800,1791.836200,1.352467e+08
std,NaN,168.078413,171.236758,165.929170,169.102646,1.397297e+07


now the high/ close are balanced and more stable

## LOG 3 Fix

 rows (out of 26,100) had high/close values ~11x too large relative to open/low and neighboring days , most probably  a data entry/generation bug affecting a specific subset of records.
    generalized to a systematic scan (high/close > 3x open/low max) once the pattern repeated.
    Fix - divided the affected high/close values by 10, since it consistently restored them to a value consistent with open/low at near days
note  - this is an inferred correction based on pattern-matching, not verified ground truth. based on evidence
5 of these 10 rows fell inside the June 1 – Oct 13 analysis window (FORD, EXXO, AMAZ, CISC, PFIZ) — left uncorrected, they could have produced false top-5 performers.

## LOG 4 - Stock Split Anomaly
General Electric (GENE) showed a massive 940% return, driven by a price jump from ~$22 to ~$182 overnight on Aug 2, 2021. This was a 1-for-8 reverse stock split. We must adjust historical prices to reflect this corporate action so performance calculations remain accurate.

In [ ]:
# ── PHASE 1.5 : Corporate Actions (Stock Splits) ──────────────────────────────
# A 1-for-8 reverse split multiplies price by 8 and divides volume by 8.
mask_gene = (df["ticker"] == "GENE") & (df["date"] < "2021-08-02")

df.loc[mask_gene, ["open", "high", "low", "close"]] *= 8
df.loc[mask_gene, "volume"] /= 8

print("Adjusted General Electric historical prices for 1-for-8 reverse split.")

## LOG 4 - Stock Split Anomaly
General Electric (GENE) showed a massive 940% return, driven by a price jump from ~$22 to ~$182 overnight on Aug 2, 2021. This was a 1-for-8 reverse stock split. We must adjust historical prices to reflect this corporate action so performance calculations remain accurate.

In [ ]:
# ── PHASE 1.5 : Corporate Actions (Stock Splits) ──────────────────────────────
# A 1-for-8 reverse split multiplies price by 8 and divides volume by 8.
mask_gene = (df["ticker"] == "GENE") & (df["date"] < "2021-08-02")

df.loc[mask_gene, ["open", "high", "low", "close"]] *= 8
df.loc[mask_gene, "volume"] /= 8

print("Adjusted General Electric historical prices for 1-for-8 reverse split.")

# PHASE 2 — Performance Calculation (Tasks 1 & 2)

Filter to the analysis window **Jun 1 – Oct 13, 2021**, get the first and last available close price per stock, compute % return, and extract the Top 5.

In [47]:
# ── PHASE 2 : Performance Calculation ────────────────────────────────────────
START_DATE = "2021-06-01"
END_DATE   = "2021-10-13"

# Filter to the analysis window
window = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()

# For each stock, get the FIRST available close price on or after START_DATE
# and the LAST available close price on or before END_DATE.
# Using groupby + sort — not every stock may trade on exactly Jun 1.
start_prices = (
    window.sort_values("date")
          .groupby("isin")
          .first()          # first row in window = earliest close on/after Jun 1
          [["name", "ticker", "close"]]
          .rename(columns={"close": "start_close"})
)

end_prices = (
    window.sort_values("date")
          .groupby("isin")
          .last()           # last row in window = closest close on/before Oct 13
          [["close"]]
          .rename(columns={"close": "end_close"})
)

# Merge and compute % return
perf = start_prices.join(end_prices)
perf["return_pct"] = (perf["end_close"] - perf["start_close"]) / perf["start_close"] * 100

# Top 5
top5 = perf.nlargest(5, "return_pct").reset_index()

print("=== Top 5 Performing Stocks (Jun 1 – Oct 13, 2021) ===\n")
print(top5[["name", "ticker", "isin", "start_close", "end_close", "return_pct"]].to_string(index=False))


=== Top 5 Performing Stocks (Jun 1 – Oct 13, 2021) ===

            name ticker         isin  start_close  end_close  return_pct
General Electric   GENE XX0000000010      19.6640   204.5057  940.000509
         Netflix   NETF XX0000000028     245.6627   687.8557  180.000057
           Adobe   ADOB XX0000000036     424.4101  1026.4742  141.859042
           Nokia   NOKI XX0000000044       1.4053     2.9512  110.004981
          NVIDIA   NVID XX0000000051      72.1659   133.5070   85.000118


In [49]:
df[df["ticker"] == "GENE"].sort_values("date").query("date >= '2021-05-25' and date <= '2021-06-05'")

,date,isin,ticker,name,open,high,low,close,volume
10020,2021-05-25,XX0000000010,GENE,General Electric,20.0384,20.4453,19.7736,20.3897,46217426.0
10021,2021-05-26,XX0000000010,GENE,General Electric,20.4551,20.6524,20.1340,20.4181,53272285.0
10022,2021-05-27,XX0000000010,GENE,General Electric,20.1414,20.2185,19.5448,19.6492,58424545.0
10023,2021-05-28,XX0000000010,GENE,General Electric,19.5862,20.2639,19.1743,20.0342,63041675.0
10024,2021-05-31,XX0000000010,GENE,General Electric,20.1052,20.2577,19.8871,19.9284,72652291.0
10025,2021-06-01,XX0000000010,GENE,General Electric,20.1448,20.2231,19.6254,19.6640,87582375.0
10026,2021-06-02,XX0000000010,GENE,General Electric,19.7008,19.7653,18.9194,19.0915,35481706.0
10027,2021-06-03,XX0000000010,GENE,General Electric,19.1182,19.1644,18.2469,18.5057,85301395.0
10028,2021-06-04,XX0000000010,GENE,General Electric,18.3585,19.1785,18.0866,19.0996,71381181.0


In [50]:
df[df["ticker"] == "GENE"].sort_values("date").query("date >= '2021-10-08' and date <= '2021-10-18'")

,date,isin,ticker,name,open,high,low,close,volume
10118,2021-10-08,XX0000000010,GENE,General Electric,208.3721,210.8359,197.4225,202.3670,7945483.0
10119,2021-10-11,XX0000000010,GENE,General Electric,201.8420,205.6744,199.6931,203.2823,5115387.0
10120,2021-10-12,XX0000000010,GENE,General Electric,202.0740,209.6309,201.3837,207.1222,6208598.0
10121,2021-10-13,XX0000000010,GENE,General Electric,204.6801,205.2392,204.4442,204.5057,7415477.0
10122,2021-10-14,XX0000000010,GENE,General Electric,202.9308,205.9901,202.0851,205.1444,4843681.0
10123,2021-10-15,XX0000000010,GENE,General Electric,204.9941,209.9137,202.8165,208.2021,5585831.0
10124,2021-10-18,XX0000000010,GENE,General Electric,207.8437,211.8696,206.8242,209.5865,5590855.0


In [51]:
top10 = perf.nlargest(10, "return_pct").reset_index()
print(top10[["name", "ticker", "isin", "start_close", "end_close", "return_pct"]].to_string(index=False))

            name ticker         isin  start_close  end_close  return_pct
General Electric   GENE XX0000000010      19.6640   204.5057  940.000509
         Netflix   NETF XX0000000028     245.6627   687.8557  180.000057
           Adobe   ADOB XX0000000036     424.4101  1026.4742  141.859042
           Nokia   NOKI XX0000000044       1.4053     2.9512  110.004981
          NVIDIA   NVID XX0000000051      72.1659   133.5070   85.000118
 Lockheed Martin   LOCK XX0000000531     189.3405   321.5957   69.850455
          AbbVie   ABBV XX0000000754     127.2667   215.3463   69.208678
           FedEx   FEDE XX0000000978     343.6948   578.3081   68.262103
      McDonald's   MCDO XX0000000309     331.8647   556.5033   67.689815
      Home Depot   HOME XX0000000432     156.3498   259.6503   66.070120


There's a clean, decisive gap between rank 5 (NVIDIA, 85.0%) and rank 6 (Lockheed Martin, 69.85%, a ~15%point drop. That's a comfortable margin, meaning the ranking isn't sensitive to any small remaining data wobble; these 5 are clearly, unambiguously the top performers.